In [7]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Part 2: NeRF 3D Reconstruction

This notebook orchestrates Part 2 training and rendering using helper modules:
- `rendering.py`
- `nerf_model.py`
- `train_part2.py`
- `part2_utils.py`

Start with a smoke run, then scale to full training.

In [2]:
from pathlib import Path
import numpy as np
import torch

In [3]:
from part2_utils import ensure_dir, plot_training_curves, set_seed
from train_part2 import build_part2_data, render_test_trajectory_depth, render_test_trajectory_rgb, train_nerf_part2

In [10]:
# Core configuration
CFG = {
    "seed": 42,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "data_path": "lego_200x200.npz",
    "output_dir": "images/output/part2",
    "hidden_dim": 256,
    "n_layers": 8,
    "pos_freqs": 10,
    "dir_freqs": 4,
    "n_coarse": 32,
    "n_fine": 32,
    "batch_rays": 256,
    "n_steps": 200,
    "eval_every": 50,
    "chunk_size": 4096,
    "lr": 5e-4,
    "near_override": 2.0,
    "far_override": 6.0,
}

set_seed(CFG["seed"])
ensure_dir(CFG["output_dir"])
CFG

{'seed': 42,
 'device': 'cpu',
 'data_path': 'lego_200x200.npz',
 'output_dir': 'images/output/part2',
 'hidden_dim': 256,
 'n_layers': 8,
 'pos_freqs': 10,
 'dir_freqs': 4,
 'n_coarse': 32,
 'n_fine': 32,
 'batch_rays': 256,
 'n_steps': 200,
 'eval_every': 50,
 'chunk_size': 4096,
 'lr': 0.0005,
 'near_override': 2.0,
 'far_override': 6.0}

In [12]:
# Quick data sanity check (no training)
data = build_part2_data(CFG["data_path"], device=CFG["device"])
print("K shape:", tuple(data["k"].shape))
print("Train rays:", len(data["train_dataset"]))
print("Val images:", tuple(data["val_images"].shape))
print("Test poses:", tuple(data["test_c2ws"].shape))

K shape: (3, 3)
Train rays: 4000000
Val images: (10, 200, 200, 3)
Test poses: (60, 4, 4)


In [6]:
# Launch Viser server for camera/ray/sample visualization
import viser
import numpy as np
import torch
import time
from dataset_3d import load_data, RaysData
from rendering import sample_along_rays

# Load data and setup the rays specifically from the first camera for a clean visualization 
# (as recommended in the assignment to make sure rays stay within the camera frustum)
images_train, c2ws_train, images_val, c2ws_val, c2ws_test, K = load_data(data_path=CFG["data_path"])
H, W = images_train.shape[1], images_train.shape[2]

# RaysData class internally moves tensors to the given device.
# But it expects them to be properly instantiated first if the user environment 
# doesn't auto-handle numpy to torch array conversions across all internal lines.
# We'll construct it on CPU, since Torch indicates issues with CUDA compilation in this specific kernel build
device = "cpu"
images_train_t = torch.tensor(images_train, dtype=torch.float32) if isinstance(images_train, np.ndarray) else images_train
K_t = torch.tensor(K, dtype=torch.float32) if isinstance(K, np.ndarray) else K
c2ws_train_t = torch.tensor(c2ws_train, dtype=torch.float32) if isinstance(c2ws_train, np.ndarray) else c2ws_train

dataset = RaysData(images_train_t, K_t, c2ws_train_t, device=device)

# Get rays of just the first image
rays_o_first_image = dataset.rays_o[:H*W]
rays_d_first_image = dataset.rays_d[:H*W]

# Sample random rays from the first image
num_rays = 100
indices = np.random.randint(low=0, high=H * W, size=num_rays)

rays_o = rays_o_first_image[indices]
rays_d = rays_d_first_image[indices]

# Use the exact argument names from rendering.py
points, t_vals = sample_along_rays(
    ray_origins=rays_o,
    ray_directions=rays_d,
    near=float(CFG["near_override"]),
    far=float(CFG["far_override"]),
    n_samples=int(CFG["n_coarse"]),
    perturb=False,
)

# Convert to numpy for viser
rays_o_np = rays_o.cpu().detach().numpy()
rays_d_np = rays_d.cpu().detach().numpy()
points_np = points.cpu().detach().numpy()
# Use the original numpy arrays or detached tensors
images_np = images_train if isinstance(images_train, np.ndarray) else images_train.cpu().detach().numpy()
c2ws_np = c2ws_train if isinstance(c2ws_train, np.ndarray) else c2ws_train.cpu().detach().numpy()
K_np = K if isinstance(K, np.ndarray) else K.cpu().detach().numpy()

server = viser.ViserServer(port=8080, share=False)

fov = float(2 * np.arctan2(H / 2, K_np[0, 0]))
aspect = float(W / H)

# Add all cameras
for i, (image, c2w) in enumerate(zip(images_np, c2ws_np)):
    image_uint8 = (np.clip(image, 0.0, 1.0) * 255.0).astype(np.uint8)
    server.scene.add_camera_frustum(
        f"/cameras/{i}",
        fov=fov,
        aspect=aspect,
        scale=0.15,
        wxyz=viser.transforms.SO3.from_matrix(c2w[:3, :3]).wxyz,
        position=c2w[:3, 3],
        image=image_uint8,
    )

# Add rays
for i, (o, d) in enumerate(zip(rays_o_np, rays_d_np)):
    positions = np.stack((o, o + d * CFG["far_override"]))
    server.scene.add_spline_catmull_rom(
        f"/rays/{i}",
        positions=positions,
    )

# Add point cloud (samples)
server.scene.add_point_cloud(
    "/samples",
    colors=np.zeros_like(points_np).reshape(-1, 3),
    points=points_np.reshape(-1, 3),
    point_size=0.03,
)

print("Viser server running on http://localhost:8080")
print("Run the next cell when you're done to stop the server.")

╭────── viser (listening *:8080) ───────╮
│             ╷                         │
│   HTTP      │ http://localhost:8080   │
│   Websocket │ ws://localhost:8080     │
│             ╵                         │
╰───────────────────────────────────────╯

Viser server running on http://localhost:8080
Run the next cell when you're done to stop the server.


In [7]:
# Stop Viser server launched from this notebook
try:
    server.stop()
    print("Viser server stopped.")
except Exception as e:
    print("Viser server couldn't be stopped or wasn't running:", e)

(viser) Server stopped

Viser server stopped.


In [13]:
# Smoke training run (increase n_steps and batch_rays later)
results = train_nerf_part2(
    data_path=CFG["data_path"],
    output_dir=CFG["output_dir"],
    device=CFG["device"],
    seed=CFG["seed"],
    hidden_dim=CFG["hidden_dim"],
    n_layers=CFG["n_layers"],
    pos_freqs=CFG["pos_freqs"],
    dir_freqs=CFG["dir_freqs"],
    n_coarse=CFG["n_coarse"],
    n_fine=CFG["n_fine"],
    n_steps=CFG["n_steps"],
    batch_rays=CFG["batch_rays"],
    lr=CFG["lr"],
    eval_every=CFG["eval_every"],
    chunk_size=CFG["chunk_size"],
    near_override=CFG["near_override"],
    far_override=CFG["far_override"],
)
results["metrics"]

[train_nerf_part2] start device=cpu steps=200 batch_rays=256 coarse=32 fine=32 near=2.000 far=6.000 eval_every=50
[train_nerf_part2] step 1/200 loss=0.200121 coarse=0.062844 fine=0.193837 iter=1.004s rays_per_sec=255
[train_nerf_part2] step 10/200 loss=0.152161 coarse=0.175387 fine=0.134622 iter=0.854s rays_per_sec=300
[train_nerf_part2] step 20/200 loss=0.082392 coarse=0.073057 fine=0.075087 iter=0.898s rays_per_sec=285
[train_nerf_part2] step 30/200 loss=0.080684 coarse=0.079921 fine=0.072692 iter=0.810s rays_per_sec=316
[train_nerf_part2] step 40/200 loss=0.052893 coarse=0.051382 fine=0.047755 iter=0.867s rays_per_sec=295
[train_nerf_part2] step 50/200 loss=0.070634 coarse=0.066179 fine=0.064016 iter=0.829s rays_per_sec=309
[train_nerf_part2] eval step 50/200 val_psnr=12.069dB best=12.069dB (new best, saved checkpoint) eval=53.97s
[train_nerf_part2] step 60/200 loss=0.072634 coarse=0.066548 fine=0.065979 iter=0.841s rays_per_sec=304
[train_nerf_part2] step 70/200 loss=0.057906 coars

{'best_psnr': 16.740236282348633,
 'best_step': 200,
 'final_loss': 0.01637938991189003,
 'near': 2.0,
 'far': 6.0,
 'n_steps': 200,
 'batch_rays': 256,
 'n_coarse': 32,
 'n_fine': 32,
 'val_psnr_hist': [12.06944465637207,
  13.465744256973267,
  16.33725643157959,
  16.740236282348633],
 'eval_steps': [50, 100, 150, 200],
 'total_seconds': 382.6816620999962,
 'avg_seconds_per_step': 1.9134083104999808,
 'log_every': 10}

In [14]:
# Save training curves
curve_dir = Path(CFG["output_dir"]) / "curves"
plot_training_curves(
    loss_hist=results["loss_hist"],
    eval_steps=results["eval_steps"],
    val_psnr_hist=results["val_psnr_hist"],
    output_dir=curve_dir,
)
print("Saved curves to", curve_dir)

Saved curves to images\output\part2\curves


In [15]:
%load_ext autoreload
%autoreload 2

from train_part2 import build_part2_data, render_test_trajectory_depth, render_test_trajectory_rgb, train_nerf_part2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [18]:
# Render RGB test views to NPY, PNG, and GIF
models = results["models"]
data = results["data"]
image_hw = (data["val_images"].shape[1], data["val_images"].shape[2])

render_test_trajectory_rgb(
    model_coarse=models["coarse"],
    model_fine=models["fine"],
    k=data["k"],
    test_c2ws=data["test_c2ws"],
    image_hw=image_hw,
    output_dir=CFG["output_dir"],
    near=results["metrics"]["near"],
    far=results["metrics"]["far"],
    n_coarse=CFG["n_coarse"],
    n_fine=CFG["n_fine"],
    chunk_size=CFG["chunk_size"],
    device=CFG["device"],
)
print("Saved RGB test trajectory outputs.")

Rendering test view 1/60...
Rendering test view 2/60...
Rendering test view 3/60...
Rendering test view 4/60...
Rendering test view 5/60...
Rendering test view 6/60...
Rendering test view 7/60...
Rendering test view 8/60...
Rendering test view 9/60...
Rendering test view 10/60...
Rendering test view 11/60...
Rendering test view 12/60...
Rendering test view 13/60...
Rendering test view 14/60...
Rendering test view 15/60...
Rendering test view 16/60...
Rendering test view 17/60...
Rendering test view 18/60...
Rendering test view 19/60...
Rendering test view 20/60...
Rendering test view 21/60...
Rendering test view 22/60...
Rendering test view 23/60...
Rendering test view 24/60...
Rendering test view 25/60...
Rendering test view 26/60...
Rendering test view 27/60...
Rendering test view 28/60...
Rendering test view 29/60...
Rendering test view 30/60...
Rendering test view 31/60...
Rendering test view 32/60...
Rendering test view 33/60...
Rendering test view 34/60...
Rendering test view 35/

In [ ]:
# Render depth test views to NPY, PNG, and GIF
render_test_trajectory_depth(
    model_coarse=models["coarse"],
    model_fine=models["fine"],
    k=data["k"],
    test_c2ws=data["test_c2ws"],
    image_hw=image_hw,
    output_dir=CFG["output_dir"],
    near=results["metrics"]["near"],
    far=results["metrics"]["far"],
    n_coarse=CFG["n_coarse"],
    n_fine=CFG["n_fine"],
    chunk_size=CFG["chunk_size"],
    device=CFG["device"],
)
print("Saved depth test trajectory outputs.")

In [11]:
# Preview RGB trajectory GIF
from IPython.display import Image, display

gif_path = Path(CFG["output_dir"]) / "test_rgb.gif"

if gif_path.exists():
    display(Image(data=open(gif_path, "rb").read(), format='gif'))
else:
    print("test_rgb.gif not found.")

test_rgb.gif not found.
